# Training Dataset Audit

This notebook reproduces the dataset preparation path used by training in `/home/liondungl/projects/qcircuit-generation` and audits the results.

It focuses on the two places that matter for your suspicion:

- raw per-qubit dataset properties against the paper SRV setup
- the difference between `padding_mode=max` and `padding_mode=bucket`

In particular, it compares:

- the `z` metadata inferred for bucket mode from saved tensors
- the `z` that max-padding preprocessing would derive from actual gate occupancy
- the final prepared tensors and dataloader batch shapes

In [1]:
import sys
import os
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from IPython.display import display

from notebooks.training_dataset_audit_helper import (
    DEFAULT_DATASET_ROOT,
    DEFAULT_TRAINING_CFG,
    build_loader_config,
    load_prepared_dataset,
    maybe_dataframe,
    preview_dataloaders,
    resolve_dataset_dirs,
    run_full_audit,
    summarize_prepared_dataset,
    summarize_raw_dataset,
)


In [2]:
DATASET_ROOT = Path(DEFAULT_DATASET_ROOT)
TRAINING_CFG = Path(DEFAULT_TRAINING_CFG)
BATCH_SIZE = None
SPLIT_RATIO = 0.1

print('DATASET_ROOT =', DATASET_ROOT)
print('TRAINING_CFG =', TRAINING_CFG)
print('LOADER_CFG =', build_loader_config(TRAINING_CFG, batch_size=BATCH_SIZE))

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f'Dataset root does not exist: {DATASET_ROOT}\n'
        'Point DATASET_ROOT to your local copy of the official dataset.'
    )


DATASET_ROOT = /workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit
TRAINING_CFG = /workspace/qcircuit-generation/conf/training/paper_srv_bucket.yaml
LOADER_CFG = {'training': {'padding_mode': 'bucket', 'batch_size': 256}}


In [3]:
dataset_dirs = resolve_dataset_dirs(DATASET_ROOT)
raw_summaries = [summarize_raw_dataset(path) for path in dataset_dirs]

raw_table = []
for item in raw_summaries:
    raw_table.append({
        'dataset_dir': Path(item['dataset_dir']).name,
        'num_samples_saved': item['num_samples_saved'],
        'tensor_shape': item['tensor_shape'],
        'num_qubits_config': item['num_qubits_config'],
        'min_gates_config': item['min_gates_config'],
        'max_gates_config': item['max_gates_config'],
        'matches_paper_gate_pool': item['matches_paper_gate_pool'],
        'matches_paper_min_gates': item['matches_paper_min_gates'],
        'matches_paper_max_gates': item['matches_paper_max_gates'],
        'gate_length_min': item['gate_length_min'],
        'gate_length_median': item['gate_length_median'],
        'gate_length_max': item['gate_length_max'],
        'bucket_z_time_min': item['bucket_z_time_min'],
        'bucket_z_time_max': item['bucket_z_time_max'],
        'max_stage_z_time_min': item['max_stage_z_time_min'],
        'max_stage_z_time_max': item['max_stage_z_time_max'],
        'bucket_vs_max_time_mismatch': item['bucket_vs_max_time_mismatch'],
        'bucket_time_is_constant': item['bucket_time_is_constant'],
        'prompt_unique_count': item['prompt_unique_count'],
    })

display(maybe_dataframe(raw_table))


[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_x.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_y.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_z.pt` onto device: cpu.
[INFO]: Instantiated config_dataset from given config on cpu.


,dataset_dir,num_samples_saved,tensor_shape,num_qubits_config,min_gates_config,max_gates_config,matches_paper_gate_pool,matches_paper_min_gates,matches_paper_max_gates,gate_length_min,gate_length_median,gate_length_max,bucket_z_time_min,bucket_z_time_max,max_stage_z_time_min,max_stage_z_time_max,bucket_vs_max_time_mismatch,bucket_time_is_constant,prompt_unique_count
0,qc_srv_dataset_3to8qubit,2463286,"(2463286, 8, 52)",8,2,52,False,False,True,0,17.0,52,4,52,4,52,0,False,471


In [4]:
for item in raw_summaries:
    name = Path(item['dataset_dir']).name
    print(f'\n=== {name} ===')
    print('Top prompts:')
    display(maybe_dataframe(item['top_prompts']))
    print('Entanglement buckets:')
    display(maybe_dataframe(item['entanglement_buckets']))
    print('Bucket-inferred z time counts:')
    display(maybe_dataframe(item['z_bucket_time_counts']))
    print('Max-stage z time counts:')
    display(maybe_dataframe(item['z_max_time_counts']))



=== qc_srv_dataset_3to8qubit ===
Top prompts:


,prompt,count
0,"Generate SRV: [2, 2, 2, 2]",81962
1,"Generate SRV: [2, 2, 2, 2, 2]",79881
2,"Generate SRV: [2, 2, 2]",66349
3,"Generate SRV: [1, 1, 1, 1]",53284
4,"Generate SRV: [1, 1, 1]",52605
5,"Generate SRV: [2, 2, 2, 2, 2, 2]",48874
6,"Generate SRV: [1, 1, 1, 1, 1]",37062
7,"Generate SRV: [2, 2, 2, 2, 2, 2, 2]",36503
8,"Generate SRV: [2, 2, 2, 1, 2]",33095
9,"Generate SRV: [2, 2, 1, 2, 2]",32749


Entanglement buckets:


""


Bucket-inferred z time counts:


,time,count
0,4,32140
1,8,313833
2,12,401945
3,16,415482
4,20,321918
5,24,265908
6,28,198349
7,32,174220
8,36,144810
9,40,106563


Max-stage z time counts:


,time,count
0,4,32140
1,8,313833
2,12,401945
3,16,415482
4,20,321918
5,24,265908
6,28,198349
7,32,174220
8,36,144810
9,40,106563


In [5]:
loader_max, max_dataset = load_prepared_dataset(
    DATASET_ROOT,
    training_cfg_path=TRAINING_CFG,
    padding_mode='max',
    batch_size=BATCH_SIZE,
)
loader_bucket, bucket_dataset = load_prepared_dataset(
    DATASET_ROOT,
    training_cfg_path=TRAINING_CFG,
    padding_mode='bucket',
    batch_size=BATCH_SIZE,
)

max_summary = summarize_prepared_dataset(max_dataset, padding_mode='max')
bucket_summary = summarize_prepared_dataset(bucket_dataset, padding_mode='bucket')

print('MAX PREPARATION')
pprint(max_summary)
print('\nBUCKET PREPARATION')
pprint(bucket_summary)


2026-03-23 10:33:10 - quantum_diffusion.data.dataset - INFO - Detected preprocessed dataset. Loading directly...
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_x.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_y.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_z.pt` onto device: cpu.
[INFO]: Instantiated config_dataset from given config on cpu.
2026-03-23 10:33:21 - quantum_diffusion.data.dataset - INFO - Dataset loaded from /workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit
2026-03-23 10:33:21 - quantum_diffusion.data.dataset - INFO - Detected preprocessed dataset. Loading directly...
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_x.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-

  0%|          | 0/1 [00:00<?, ?it/s]

 - balance_tensor_dataset, njobs=1, number of samples=2463286
 - uniquify_tensor_dataset, number of samples now 2463286
 - balancing


  0%|          | 0/471 [00:00<?, ?it/s]

 - dataset size after balancing 2463286
Split: Train 9141 - Test 481 

2026-03-23 10:34:07 - quantum_diffusion.data.dataset - INFO - Datasets combined into a mixed dataset
MAX PREPARATION
{'bucket_batch_size': -1,
 'collate_fn': None,
 'dataset_type': 'CircuitsConfigDataset',
 'gate_length_max': 52,
 'gate_length_min': 0,
 'has_z': True,
 'nonpad_qubits_max': 8,
 'nonpad_qubits_min': 3,
 'nonpad_time_max': 52,
 'nonpad_time_min': 1,
 'num_samples': 2463286,
 'pad_constant': 3,
 'padding_mode': 'max',
 'params': {'dataset_to_gpu': False,
            'max_gates': 52,
            'min_gates': 2,
            'num_of_qubits': 8,
            'random_samples': 0},
 'x_shape': (2463286, 8, 52),
 'z_qubits_unique': [3, 4, 5, 6, 7, 8],
 'z_shape': (2463286, 2),
 'z_time_multiple_of_scale_factor': False,
 'z_time_unique': [1,
                   2,
                   3,
                   4,
                   5,
                   6,
                   7,
                   8,
                   

In [6]:
effective_batch_size = build_loader_config(TRAINING_CFG, batch_size=BATCH_SIZE)['training']['batch_size']

preview_max = preview_dataloaders(
    loader_max,
    max_dataset,
    batch_size=effective_batch_size,
    split_ratio=SPLIT_RATIO,
)
preview_bucket = preview_dataloaders(
    loader_bucket,
    bucket_dataset,
    batch_size=effective_batch_size,
    split_ratio=SPLIT_RATIO,
)

print('MAX LOADER PREVIEW')
pprint(preview_max)
print('\nBUCKET LOADER PREVIEW')
pprint(preview_bucket)


[INFO]: Not balancing dataset!  balance_max=None
[INFO]: Generate cache: converting tensors to str and tokenize
 - to str list
 - tokenize_and_push_to_device
Tokenizing with 1 job(s)
2026-03-23 10:34:31 - quantum_diffusion.data.dataset - ERROR - Error creating dataloaders: 'NoneType' object has no attribute 'njobs'


AttributeError: 'NoneType' object has no attribute 'njobs'

In [ ]:
full_audit = run_full_audit(
    DATASET_ROOT,
    training_cfg_path=TRAINING_CFG,
    batch_size=BATCH_SIZE,
    split_ratio=SPLIT_RATIO,
)
full_audit.keys()


[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_x.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_y.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_z.pt` onto device: cpu.
[INFO]: Instantiated config_dataset from given config on cpu.


## What to look for

- If `bucket_vs_max_time_mismatch` is large, bucket mode is not using the same notion of per-sample circuit length as max-padding preprocessing.
- If `bucket_z_time_is_constant` is `True` for a raw dataset while `gate_length_*` varies a lot, bucket inference is effectively collapsing all samples in that dataset to the configured rectangular tensor length.
- In bucket mode, `bucket_qubits_all_uniform` should be `True`; otherwise the grouping by qubit count is broken.
- The dataloader preview should show bucket mode using loader batch size `1`, with the collate function producing tensors already cut to the bucket-local `z` maxima.
- For the paper SRV setup, the gate pool should stay `['h', 'cx']`, and the per-qubit `min_gates`/`max_gates` should match Extended Data Table I.